# Check digestion output for isoform specific peptides 
In this notebook we will use the results from the tryptic in silico digestion and check for how many isoforms we can find a unique peptide right after the digestion.

# Setup
## Import and install Packages

In [ ]:
import sys
import os
import session_info

In [ ]:
import pandas as pd
import ast

In [ ]:
# Display session information
session_info.show()

## Set working directory

In [ ]:
cwd = os.getcwd()
if not cwd.endswith("Isoform_Database/Jupyter_environment/Isoform_Database_SIAF/02_In-Silico_Digestion"):
    print("WARNING: The working directory is not set to the '02_In-Silico_Digestion'!")
    print("Current working directory:", cwd)
else:
    print(f"Working directory is correctly set to the '02_In-Silico_Digestion' folder (\"{cwd}\").")

In [ ]:
# Data directories
mapping_dir = "../01_UniProt/data/mapping/"
digestion_output_dir = "data/digestion_output"

## Read in Dataframes

In [ ]:
iso_canonical_mapping = pd.read_csv(os.path.join(mapping_dir, 'iso_canonical_mapping.csv'))
iso_canonical_mapping_flat = pd.read_csv(os.path.join(mapping_dir, 'iso_canonical_mapping_flat.csv'))

digestion_Trypsin_filtered =  pd.read_csv(os.path.join(digestion_output_dir, 'digestion_Trypsin_filtered.csv'))
digestion_TrypsinP_filtered =  pd.read_csv(os.path.join(digestion_output_dir, 'digestion_TrypsinP_filtered.csv'))

# Unique Isoform Peptide Fragments

Implementation: Sequential Search
In Round 1, a peptide is unique if it exists in the isoform but not in the canonical parent or the rest of the proteome. In Round 2, it must be unique to the second isoform relative to the canonical parent, the rest of the proteome, and the first isoform.

In [ ]:
def find_sequential_markers(digestion_df, mapping_df, all_isoform_ids):
    """
    Identifies sequential markers by excluding the background proteome, 
    the canonical parent, and previously processed isoforms.
    """
    # 1. Prepare global background (everything EXCEPT the specific isoforms we are testing)
    background_proteome = digestion_df[~digestion_df['ID'].isin(all_isoform_ids)]
    background_peptides = set(background_proteome['seq'].unique())

    # 2. Pre-process peptides for quick lookup
    pep_dict = digestion_df.groupby('ID')['seq'].apply(set).to_dict()

    final_results = []

    # 3. Iterate through the Mapping (Canonical -> List of Isoforms)
    for _, row in mapping_df.iterrows():
        canonical_id = row['Entry']
        # Handle literal_eval if the isoforms are stored as strings
        isoform_list = row['Isoforms']
        if isinstance(isoform_list, str):
            isoform_list = ast.literal_eval(isoform_list)
        
        # The 'Forbidden Set' starts with the background proteome
        forbidden_peptides = background_peptides.copy()
        
        # Add the canonical parent to the forbidden set if it exists in this digestion
        if canonical_id in pep_dict:
            forbidden_peptides.update(pep_dict[canonical_id])
        
        # 4. Sequential Round Loop
        for idx, iso_id in enumerate(isoform_list):
            current_iso_peptides = pep_dict.get(iso_id, set())
            
            # A peptide is unique in this round if it's not in the forbidden set
            unique_to_round = current_iso_peptides - forbidden_peptides
            
            if unique_to_round:
                best_peptide = max(unique_to_round, key=len)
                status = f"Unique Marker Found (Round {idx+1})"
            else:
                best_peptide = None
                status = "Shared with Canonical or Previous Isoforms"
                
            final_results.append({
                "Canonical_ID": canonical_id,
                "Isoform_ID": iso_id,
                "Round": idx + 1,
                "Unique_Peptide": best_peptide,
                "Status": status,
                "Peptide_Count": len(unique_to_round)
            })
            
            # Update forbidden set: Next isoform must be unique relative to this one
            forbidden_peptides.update(current_iso_peptides)

    return pd.DataFrame(final_results)

# --- EXECUTION ---

# Define the set of all isoform IDs once
all_isoform_ids = set(iso_canonical_mapping_flat['Isoform_ID'].unique())

# Generate the two dataframes
sequential_results_Try = find_sequential_markers(digestion_Trypsin_filtered, iso_canonical_mapping, all_isoform_ids)
sequential_results_TryP = find_sequential_markers(digestion_TrypsinP_filtered, iso_canonical_mapping, all_isoform_ids)

In [ ]:
sequential_results_Try

In [ ]:
print(f"--- Summary Statistics ---")
print(sequential_results_Try['Status'].value_counts())

In [ ]:
print(f"--- Summary Statistics ---")
print(sequential_results_TryP['Status'].value_counts())

In [ ]:
sequential_results_Try[sequential_results_Try['Status'].str.contains("Unique Marker Found", na=False)]

In [ ]:
sequential_results_TryP[sequential_results_TryP['Status'].str.contains("Unique Marker Found", na=False)]

In [ ]:
sequential_results_Try.to_csv('data/sequential_results_Try.csv', index = False)
sequential_results_TryP.to_csv('data/sequential_results_TryP.csv', index = False)

In [ ]:
sequential_results_Try['Isoform_ID'].nunique()

In [ ]:
iso_canonical_mapping_flat['Isoform_ID'].nunique()

In [ ]:
# Filter for Round 1 only
round_1_results = sequential_results_Try[sequential_results_Try['Round'] == 1]

# Get the breakdown of Status
print("Round 1 Marker Summary Trypsin:")
print(round_1_results['Status'].value_counts())

# Calculate the success percentage
total_round_1 = len(round_1_results)
found_count_1 = round_1_results['Status'].str.contains("Unique Marker Found", na=False).sum()

print(f"\nSuccess Rate: {(found_count_1 / total_round_1) * 100:.2f}%")
print("-" * 30)

# Filter for Round 2 only
round_2_results = sequential_results_Try[sequential_results_Try['Round'] == 2]

# Get the breakdown of Status
print("Round 2 Marker Summary Trypsin:")
print(round_2_results['Status'].value_counts())

# Calculate the success percentage
total_round_2 = len(round_2_results)
found_count_2 = round_2_results['Status'].str.contains("Unique Marker Found", na=False).sum()

print(f"\nSuccess Rate: {(found_count_2 / total_round_2) * 100:.2f}%")

In [ ]:
# Filter for Round 1 only
round_1_results = sequential_results_TryP[sequential_results_TryP['Round'] == 1]

# Get the breakdown of Status
print("Round 1 Marker Summary Trypsin_p:")
print(round_1_results['Status'].value_counts())

# Calculate the success percentage
total_round_1 = len(round_1_results)
found_count_1 = round_1_results['Status'].str.contains("Unique Marker Found", na=False).sum()

print(f"\nSuccess Rate: {(found_count_1 / total_round_1) * 100:.2f}%")
print("-" * 30)

# Filter for Round 2 only
round_2_results = sequential_results_TryP[sequential_results_TryP['Round'] == 2]

# Get the breakdown of Status
print("Round 2 Marker Summary Trypsin_P:")
print(round_2_results['Status'].value_counts())

# Calculate the success percentage
total_round_2 = len(round_2_results)
found_count_2 = round_2_results['Status'].str.contains("Unique Marker Found", na=False).sum()

print(f"\nSuccess Rate: {(found_count_2 / total_round_2) * 100:.2f}%")